# Apply Author-Name Curations

Applies author-name curations from `openalex.authors.author_names_curations` onto `openalex.authors.authors` (the profile-level source-of-truth table read by `CreateAuthors`).

Runs in `jobs/authors.yaml` after `SyncAuthorNameCurations` and before `CreateAuthors`. The downstream chain after this:

```
ApplyAuthorNameCurations → CreateAuthors → openalex_authors
                          ↓ enqueue work_ids
                          curated_work_ids_pending_sync
                          ↓ next end2end cycle
                          UpdateWorkAuthorships re-emits works
                          → sync_works → ES → API
```

## Apply-every-cycle semantics

Apply runs every cycle (daily, with the `Authors` job). No `applied_at`, no per-row state. The MERGE is gated by a no-op churn guard: rows where the curated value already matches the current `authors` value are skipped. Idempotent — second-and-later runs with no source change do zero writes and zero enqueues.

## Diff-first pattern

We materialize the diff (curations whose values differ from current `authors` rows) into a real intermediate table `openalex.authors.author_names_pending_changes`. Both the work-id enqueue and the `authors` MERGE read from that single table, so the two side effects can't disagree on "what changed this run." This is the entity-level template the user-curation pattern reuses: **diff → enqueue affected work_ids → MERGE into entity source**. Reusable for institution/source/topic curation when those land.

## Enqueue only on `display_name` change

Curated `display_name` is denormalized inside every work's `authorships[i].author.display_name`, so changing it requires the work doc to re-emit. We enqueue every `work_id` where the affected author appears in `work_authors`, with `curated_ras=NULL` (the existing convention for author-curation re-flows).

Curated `full_name` is *not* denormalized in works — it only feeds matching via `block_key` + the parsed first/middle/last. A `full_name` change reshapes `authors_for_matching.block_key` on the next `CreateAuthors` run, which `MatchAuthors` picks up automatically on the next end2end cycle. No work-level enqueue needed for `full_name`-only changes.

## Stage diff (which curations will actually change something)

In [ ]:
%sql
-- Stage which curations will actually change something (no-op churn guard).
-- Null-safe inequality via NOT (a <=> b) so NULL display_name vs curated 'X' is detected.
-- Only authors that exist in openalex.authors.authors are eligible (LEFT JOIN drops curations
-- targeting unknown author_ids; should never happen but defensive).
CREATE OR REPLACE TABLE openalex.authors.author_names_pending_changes AS
SELECT
  c.author_id,
  c.curated_display_name,
  c.curated_full_name,
  (c.curated_display_name IS NOT NULL AND NOT (a.display_name <=> c.curated_display_name)) AS display_name_changed,
  (c.curated_full_name    IS NOT NULL AND NOT (a.full_name    <=> c.curated_full_name))    AS full_name_changed
FROM openalex.authors.author_names_curations c
JOIN openalex.authors.authors a ON a.id = c.author_id
WHERE (c.curated_display_name IS NOT NULL AND NOT (a.display_name <=> c.curated_display_name))
   OR (c.curated_full_name    IS NOT NULL AND NOT (a.full_name    <=> c.curated_full_name))

## Enqueue affected work_ids for ES re-sync

In [ ]:
%sql
-- Enqueue every work_id where a display_name-changed author appears in work_authors.
-- Skipped when only full_name changed (full_name isn't denormalized in works).
-- curated_ras=NULL marks this as an author-curation re-flow (matches ApplyWorkAuthorCurations convention).
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT wa.work_id, CAST(NULL AS BOOLEAN) AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.works.work_authors wa
JOIN openalex.authors.author_names_pending_changes p ON wa.author_id = p.author_id
WHERE p.display_name_changed

## Apply curated values to `openalex.authors.authors`

In [ ]:
%sql
-- MERGE curated values into the profile-level source-of-truth. CreateAuthors reads from this
-- table on its next run and rolls the curated display_name/full_name into openalex_authors.
-- COALESCE preserves the existing value when a property wasn't curated (e.g., only display_name
-- changed for this author). updated_date bumps so downstream consumers that key off it can notice.
MERGE INTO openalex.authors.authors AS target
USING openalex.authors.author_names_pending_changes AS source
ON target.id = source.author_id
WHEN MATCHED THEN
  UPDATE SET
    target.display_name = COALESCE(source.curated_display_name, target.display_name),
    target.full_name    = COALESCE(source.curated_full_name,    target.full_name),
    target.updated_date = current_timestamp()

## Verify

In [ ]:
%sql
-- Summary: how many curations are present, how many actually changed this run,
-- and how many work_ids were enqueued.
SELECT
  (SELECT COUNT(*) FROM openalex.authors.author_names_curations)         AS total_curations,
  (SELECT COUNT(*) FROM openalex.authors.author_names_pending_changes)   AS pending_changes,
  (SELECT COUNT_IF(display_name_changed)
     FROM openalex.authors.author_names_pending_changes)                 AS display_name_changes,
  (SELECT COUNT_IF(full_name_changed)
     FROM openalex.authors.author_names_pending_changes)                 AS full_name_changes,
  (SELECT COUNT(DISTINCT wa.work_id)
     FROM openalex.works.work_authors wa
     JOIN openalex.authors.author_names_pending_changes p
       ON wa.author_id = p.author_id
     WHERE p.display_name_changed)                                       AS works_enqueued

In [ ]:
%sql
-- Spot-check: curated authors and the resulting values in openalex.authors.authors.
SELECT
  c.author_id,
  c.curated_display_name,
  a.display_name AS authors_display_name,
  c.curated_full_name,
  a.full_name AS authors_full_name,
  a.updated_date
FROM openalex.authors.author_names_curations c
LEFT JOIN openalex.authors.authors a ON a.id = c.author_id
ORDER BY a.updated_date DESC NULLS LAST
LIMIT 100